# Compute Aurora Asset Exposure

Computes **`og.fact_asset_exposure`** (per-asset weather-damage exposure scores) from the **live Aurora
forecast** instead of the seeded HURDAT2 track used by `generate_onegrid_data`.

Pipeline: read the Aurora `model-outputs` weather events (published by `aurora_pipeline/publish.py`)
-> rebuild `dim_weather_event` + `WeatherForecast` -> run the risk-engine exposure scoring
(port of `risk-engine.ts`, identical to `generate_onegrid_data`) against `dim_asset` x `dim_site`
-> write Delta to `og.*`.

If the Aurora blob is not reachable, it falls back to recomputing exposure from the forecast
already in the gold `og.WeatherForecast` / `og.dim_weather_event` tables, so the notebook always runs.


In [ ]:
# CELL: Parameters  (tag as "Parameters" in Fabric)
SCHEMA      = "og"
WRITE_MODE  = "overwrite"

# --- deploy rebind: these GUIDs are find/replaced with the target ws/lakehouse (same as generate_onegrid_data) ---
WS = "163ba38c-3869-406f-adb7-37cbc981390c"
LH = "7e08480c-cf8d-4206-901d-38b74dbe35d9"
BASE   = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"
TABLES = f"{BASE}/Tables"

# --- Aurora forecast events (published by aurora_pipeline/publish.py -> model-outputs/weather-events.json) ---
# Primary source: a copy dropped into the lakehouse OneLake Files (read with the notebook's own identity).
ONELAKE_EVENTS_PATH = f"{BASE}/Files/aurora/weather-events.json"
# Secondary source: the model-outputs blob directly (needs DefaultAzureCredential to have blob read).
AURORA_CONTAINER_URL = "https://pcproujcb2qv6vi2ts.blob.core.windows.net/model-outputs"
AURORA_BLOB_NAME     = "weather-events.json"   # config.output_blob_name (matches the Container Apps job)
USE_AURORA_BLOB      = True          # False -> recompute from existing gold WeatherForecast/dim_weather_event


In [ ]:
# CELL: Imports + risk-engine helpers (identical constants to generate_onegrid_data)
import math, json, datetime as dt
from pyspark.sql import functions as F, types as T

EARTH_MI = 3958.8
def haversine_mi(a_lat, a_lon, b_lat, b_lon):
    dlat = math.radians(b_lat - a_lat); dlon = math.radians(b_lon - a_lon)
    x = math.sin(dlat/2)**2 + math.cos(math.radians(a_lat))*math.cos(math.radians(b_lat))*math.sin(dlon/2)**2
    return 2*EARTH_MI*math.asin(math.sqrt(min(1, max(0, x))))

def wind_at(dist, core):
    if dist <= 30: return float(core)
    return float(max(12, round(core*math.exp(-(dist-30)/95))))
def rain_at(dist, cat): return max(0.2, round((3+cat*1.9)*math.exp(-dist/180), 1))
def level_from(s): return "critical" if s>=80 else "high" if s>=62 else "elevated" if s>=42 else "monitor" if s>=22 else "normal"

TYPE_SENS = {"offshore_platform":8,"pipeline":5,"lng":6,"refinery":6,"terminal":5,"port":5,"power_plant":4,"well":2}
CRIT_PTS  = {"business_critical":10,"important":6,"standard":2}
# regions exposed to tropical systems (Aurora forecasts hurricanes / tropical storms)
HURRICANE_REGIONS = {"gulf_offshore","gulf_coast","south","northeast"}
TROPICAL_KINDS    = {"hurricane","tropical_storm"}


In [ ]:
# CELL: Load Aurora weather events (OneLake Files -> model-outputs blob -> gold fallback)
def load_events_onelake():
    """Read the Aurora events JSON dropped into the lakehouse Files area (native notebook identity)."""
    raw = spark.read.option("wholetext", True).text(ONELAKE_EVENTS_PATH).collect()[0]["value"]
    payload = json.loads(raw)
    evs = payload.get("events", payload if isinstance(payload, list) else [])
    print(f"OneLake events: {len(evs)} event(s) from {ONELAKE_EVENTS_PATH}")
    return evs

def load_aurora_events():
    """Read the events JSON published by aurora_pipeline to the model-outputs container."""
    from azure.storage.blob import BlobClient
    from azure.identity import DefaultAzureCredential
    url = f"{AURORA_CONTAINER_URL.rstrip('/')}/{AURORA_BLOB_NAME}"
    bc = BlobClient.from_blob_url(url, credential=DefaultAzureCredential())
    payload = json.loads(bc.download_blob().readall())
    evs = payload.get("events", payload if isinstance(payload, list) else [])
    print(f"Aurora blob: {len(evs)} event(s) from {url}")
    return evs

def events_from_gold():
    """Fallback: rebuild the event/forecast shape from the existing gold tables."""
    dwe = spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/dim_weather_event").collect()
    wf  = spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/weatherforecast").collect()
    fc = {}
    for r in wf:
        fc.setdefault(r["event_id"], []).append(r)
    evs = []
    for e in dwe:
        pts = sorted(fc.get(e["event_id"], []), key=lambda r: r["hour"])
        evs.append({
            "id": e["event_id"], "name": e["name"], "kind": e["hazard_kind"], "status": e["status"],
            "basin": e["region"], "currentCategory": e["category"], "currentWindMph": e["current_wind_mph"],
            "gustMph": e["gust_mph"], "pressureMb": e["pressure_mb"], "movementDeg": e["movement_deg"],
            "movementMph": e["movement_mph"], "lat": e["lat"], "lon": e["lon"], "updatedAtIso": e["updated_at"],
            "cycleId": "00Z", "forecast": [
                {"hour": p["hour"], "lat": p["lat"], "lon": p["lon"], "windMph": p["wind_mph"],
                 "coneRadiusMi": p["cone_radius_mi"], "category": p["category"], "pressureMb": p["pressure_mb"]}
                for p in pts],
        })
    print(f"Gold fallback: reconstructed {len(evs)} event(s) from og.dim_weather_event/weatherforecast")
    return evs

events = []
if ONELAKE_EVENTS_PATH:
    try:
        events = load_events_onelake()
    except Exception as e:
        print("OneLake events read failed:", e)
if not events and USE_AURORA_BLOB and AURORA_CONTAINER_URL:
    try:
        events = load_aurora_events()
    except Exception as e:
        print("Aurora blob read failed:", e)
if not events:
    events = events_from_gold()
SOURCE = "aurora" if (events and str(events[0].get("id","")).startswith("aurora")) else "gold"
print("exposure source =", SOURCE)
assert events, "No weather events available from Aurora blob or gold tables."


In [ ]:
# CELL: Rebuild dim_weather_event + WeatherForecast + fc_by_event from the events
event_rows = []; forecast_rows = []; fc_by_event = {}
for ev in events:
    eid = ev["id"]
    event_rows.append((
        eid, ev.get("kind","hurricane"), ev.get("name",""), ev.get("basin",""), ev.get("status","active"),
        int(ev.get("currentCategory",0)), int(round(ev.get("currentWindMph",0))), int(round(ev.get("gustMph",0))),
        int(round(ev.get("pressureMb",0))), float(ev["lat"]), float(ev["lon"]),
        int(round(ev.get("movementDeg",0))), float(ev.get("movementMph",0.0)), str(ev.get("updatedAtIso","")),
    ))
    for fp in ev.get("forecast", []):
        row = (eid, int(fp["hour"]), float(fp["lat"]), float(fp["lon"]), float(fp["windMph"]),
               float(fp.get("coneRadiusMi", 25.0)), int(fp.get("category",0)), int(round(fp.get("pressureMb",0))))
        forecast_rows.append(row)
        fc_by_event.setdefault(eid, []).append(row)
for eid in fc_by_event:  # forecast hours must be increasing for closest-approach interpolation
    fc_by_event[eid].sort(key=lambda r: r[1])

dim_weather_event = spark.createDataFrame(event_rows, ["event_id","hazard_kind","name","region","status",
    "category","current_wind_mph","gust_mph","pressure_mb","lat","lon","movement_deg","movement_mph","updated_at"])
WeatherForecast   = spark.createDataFrame(forecast_rows, ["event_id","hour","lat","lon","wind_mph",
    "cone_radius_mi","category","pressure_mb"])
kind_of = {e[0]: e[1] for e in event_rows}
print(f"events={len(event_rows)} forecast_points={len(forecast_rows)}")


In [ ]:
# CELL: Load assets + sites (asset geo / region / type / criticality)
sites  = {r["site_id"]: r.asDict() for r in spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/dim_site").collect()}
assets = [r.asDict() for r in spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/dim_asset").collect()]
print(f"assets={len(assets)} sites={len(sites)}")

# equipment condition (aakr_health) for the damage/fragility model; default 85 if absent
try:
    health_of = {r["asset_id"]: float(r["health_score"]) for r in spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/aakr_health").collect()}
except Exception as _e:
    health_of = {}
    print("aakr_health not available -> default health 85:", _e)
cat_of = {a["asset_id"]: (a.get("equipment_category") or "Static") for a in assets}
print(f"health rows={len(health_of)}")


In [ ]:
# CELL: Exposure scoring (port of risk-engine.ts, identical to generate_onegrid_data)
def cycle_of(ev_updated):
    # "2026-08-27T09:03:11Z" -> "09Z"; else "00Z"
    try:    return ev_updated[11:13] + "Z"
    except Exception: return "00Z"

trop_events = [e for e in event_rows if e[1] in TROPICAL_KINDS]
exp_rows = []
for a in assets:
    site = sites.get(a["site_id"])
    if not site: continue
    region = site["region"]
    if region not in HURRICANE_REGIONS: continue
    slat, slon, stype, crit = site["lat"], site["lon"], site["site_type"], site["criticality"]
    for ev in trop_events:
        eid = ev[0]; pts = fc_by_event.get(eid, [])
        if not pts: continue
        # closest approach: interpolate each segment into 6ths (exact port)
        best = None
        for i in range(len(pts)):
            samples = [(pts[i][2], pts[i][3], pts[i][1], pts[i])]
            if i+1 < len(pts):
                A, B = pts[i], pts[i+1]
                for t in range(1, 6):
                    f = t/6
                    samples.append((A[2]+(B[2]-A[2])*f, A[3]+(B[3]-A[3])*f, A[1]+(B[1]-A[1])*f, A if f<0.5 else B))
            for (plat, plon, phour, base) in samples:
                d = haversine_mi(slat, slon, plat, plon)
                if best is None or d < best[0]: best = (d, phour, base)
        dist, hour, base = best; cat = base[6]; core = base[4]; cone = base[5]
        w = wind_at(dist, core); rn = rain_at(dist, cat)
        pts_sum  = round(max(0, 26*math.exp(-dist/70)))     # proximity
        pts_sum += round(min(24, max(0, (w-50)/3.7)))       # wind
        pts_sum += round(min(8, rn*0.9))                    # rain
        pts_sum += round(max(0, 10-hour/12))                # eta
        pts_sum += 0 if dist>220 else round(cat*2)          # intensity
        pts_sum += CRIT_PTS.get(crit, 2)                    # criticality
        pts_sum += TYPE_SENS.get(stype, 4)                  # type sensitivity
        inside = dist <= max(cone, 25)
        if inside: pts_sum += 6
        score = max(0, min(100, round(pts_sum))); lvl = level_from(score)
        threat = (f"{w:.0f} mph sustained wind" if w>=74
                  else f"{rn} in rainfall" if rn>=4 else "monitoring only")
        exp_rows.append((a["asset_id"], a["site_id"], cycle_of(ev[13]), kind_of[eid], score, lvl,
                         threat, float(w), float(rn), int(round(dist)), int(round(hour)), bool(inside)))

fact_asset_exposure = spark.createDataFrame(exp_rows, ["asset_id","site_id","cycle_id","hazard_kind","score",
    "level","primary_threat","forecast_wind_mph","rainfall_in","distance_mi","hours_to_impact","inside_threat_area"])
print(f"exposure_rows={len(exp_rows)}  critical={sum(1 for r in exp_rows if r[5]=='critical')}")

# ---- Predicted equipment damage (fragility model; mirror of report-app dataApi.js DMG) ----
DMG_WIND = {"Electrical":(85,0.055),"Rotating":(100,0.05),"Fired":(105,0.05),"Static":(120,0.045)}
DMG_FLOOD = {"Electrical":0.85,"Rotating":0.55,"Fired":0.45,"Static":0.30}
DMG_REPAIR_H = {"Electrical":60,"Rotating":96,"Fired":168,"Static":120}
DMG_REPAIR_USD = {"Electrical":450000,"Rotating":1300000,"Fired":2600000,"Static":900000}
DMG_OUTAGE_USD_H = {"business_critical":22000,"important":11000,"standard":4000}
RAIN_ONSET, RAIN_SAT = 4.0, 14.0
def damage_for(asset_id, site_id, wind, rain):
    cat = cat_of.get(asset_id, "Static")
    if cat not in DMG_WIND: cat = "Static"
    w50, k = DMG_WIND[cat]
    p_wind = 1.0/(1.0+math.exp(-k*(wind-w50)))
    p_flood = DMG_FLOOD[cat]*min(1.0, max(0.0, (rain-RAIN_ONSET)/(RAIN_SAT-RAIN_ONSET)))
    p_base = 1.0-(1.0-p_wind)*(1.0-p_flood)
    hs = health_of.get(asset_id, 85.0)
    hf = 1.0+((100.0-hs)/100.0)*0.5
    p = min(0.98, p_base*hf)
    down = int(round(p*DMG_REPAIR_H[cat]))
    crit = (sites.get(site_id) or {}).get("criticality", "standard")
    outage = DMG_OUTAGE_USD_H.get(crit, 4000)
    loss = int(round((p*DMG_REPAIR_USD[cat] + down*outage)/1000.0)*1000)
    if p_flood >= p_wind and rain >= RAIN_ONSET: mode = "Flooding / water ingress"
    elif wind >= 74: mode = "Hurricane-force wind"
    elif wind >= 39: mode = "Storm-force wind / debris"
    elif p >= 0.05: mode = "Marginal wind exposure"
    else: mode = "Negligible"
    return (round(float(p),3), mode, down, loss, cat)
dmg_rows = []
for r in exp_rows:
    p, mode, down, loss, cat = damage_for(r[0], r[1], float(r[7]), float(r[8]))
    dmg_rows.append((r[0], r[1], r[3], cat, p, mode, down, loss, float(r[7]), float(r[8]), int(r[9]), int(r[10])))
fact_asset_damage = spark.createDataFrame(dmg_rows, ["asset_id","site_id","hazard_kind","equipment_category",
    "damage_probability","damage_mode","expected_downtime_h","expected_loss_usd",
    "forecast_wind_mph","rainfall_in","distance_mi","hours_to_impact"])
print(f"damage_rows={len(dmg_rows)}  worst_loss=${max([d[7] for d in dmg_rows], default=0):,}")


In [ ]:
# CELL: Write Delta (gold) + validate
def write_delta(df, name):
    dst = f"{TABLES}/{SCHEMA}/{name}"
    df.write.format("delta").mode(WRITE_MODE).option("overwriteSchema","true").save(dst)
    print("wrote", f"{SCHEMA}.{name}", "->", dst, "rows=", df.count())

write_delta(dim_weather_event, "dim_weather_event")   # refreshed from Aurora
write_delta(WeatherForecast,   "weatherforecast")     # refreshed from Aurora
write_delta(fact_asset_exposure, "fact_asset_exposure")
write_delta(fact_asset_damage, "fact_asset_damage")   # predicted physical damage per asset

# referential integrity (should be 0 orphans)
for ckey in ["asset_id","site_id"]:
    parent = "dim_asset" if ckey=="asset_id" else "dim_site"
    c = spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/fact_asset_exposure")
    p = spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/{parent}")
    print(f"fact_asset_exposure.{ckey} -> {parent}.{ckey}: {c.join(p, c[ckey]==p[ckey], 'left_anti').count()} orphans")
print("done. NOTE: run a semantic-model refresh (Import model) after this so the app sees the new exposure.")
